<a href="https://colab.research.google.com/github/jonik2909/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install transformers torch bitsandbytes datasets peft

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch
from datasets import load_dataset

In [ ]:
# 1. Configuration and Tokenizer
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

# 2. Quantization Setup (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

# 3. Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # Automatically handles GPU/CPU placement
    # dtype=torch.bfloat16
)

In [ ]:
# 4. Inference Test (Before Fine-tuning)
# prompt = "Explain what a tokenizer is."
prompt = "A tokenizer is a tool in natural language processing that"

# Prepare inputs and move to the same device as the model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

# Decode and print results
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# first_block = model.model.layers[0]
# print("first_block:", first_block)
# print(first_block.self_attn)
# print(model.config)

In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including fronzen 4-bit): {total_params:,}")

## datasets library | load_dataset
* intruction tuning

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset[0]

In [ ]:
def generate_prompt(example):
  intsruction = example['instruction']
  input_text = example['input']
  output_Text = example['output']

  if input_text:
    return (
        "### Instruction:\n"
        f"{intsruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_Text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{intsruction}\n\n"
        "### Response:\n"
        f"{output_Text}\n\n"
    )

# generate_prompt(dataset[0])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [ ]:
dataset[0]['text']

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42)

In [ ]:
dataset

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory

In [17]:
from peft import LoraConfig, get_peft_model

# Define the LoRA Configuration
lora_config = LoraConfig(
    r=8,                # Rank: lower numbers mean fewer parameters
    lora_alpha=16,      # Scaling factor for the learned weights
    lora_dropout=0.05,  # Dropout probability to prevent overfitting
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"], # Targets the Attention Query and Value matrices
)

In [18]:
# Wrap the base model with LoRA layers
model = get_peft_model(model, lora_config)

In [19]:
# Verify the reduction in trainable parameters
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
